# Figs. 3 and 4: self-similarity of the 30 DFs

The DFs raw, virial-scaled and $r_s$-scaled to Halo 2, against the Einasto DF of Halo 2
(Eddington inversion of its density fit), and the residual of each median from it.

Reads `Halo_Results/Halo_XX/df/{raw,scaled}/` and Halo 2's density fit.
Writes `fig3_{raw,virial,rs}.dat` (eps, one column per halo), `fig3_medians.dat`,
`fig3_einasto.dat` and `fig4.dat` to `output/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports

import numpy as np
import pandas as pd

from Empirical_DF_Construction.core.density_fits import einasto_shape
from Empirical_DF_Construction.core.eddington import eddington_df_from_rgrid_rho
from Empirical_DF_Construction.io.results_paths import df_dir
from Empirical_DF_Construction.paper_figures.common import EPS_HI, EPS_LO, REF_HALO, density_fit, out_dir

# log-spaced near both ends of the eps range, linear in between
EPS = np.unique(np.concatenate([
    np.logspace(np.log10(EPS_LO), np.log10(0.05), 500),
    np.linspace(0.05, 0.95, 600),
    1 - np.logspace(np.log10(0.05), np.log10(1 - EPS_HI), 500),
]))
SOURCES = {
    "raw": ("raw/df_raw_with_hubble_flow.csv", "log10f"),
    "virial": (f"scaled/df_ref{REF_HALO:02d}_with_hubble_flow_virial.csv", "log10f_ref"),
    "rs": (f"scaled/df_ref{REF_HALO:02d}_with_hubble_flow_nfw.csv", "log10f_ref"),
}
out = out_dir()

In [ ]:
def curve(path, col):
    d = pd.read_csv(path)
    x, y = d["eps_hat"].to_numpy(), d[col].to_numpy()
    k = np.isfinite(x) & np.isfinite(y) & (x >= EPS_LO) & (x <= EPS_HI)
    return np.interp(EPS, x[k], y[k])  # constant beyond the ends


curves = {name: np.array([curve(df_dir(h) / rel, col) for h in range(1, 31)])
          for name, (rel, col) in SOURCES.items()}

## Einasto reference: Eddington inversion of Halo 2's fit

In [ ]:
p = density_fit(REF_HALO)["einasto"]
r = np.logspace(np.log10(max(p["r2"] * 1e-6, 1e-12)), np.log10(p["r2"] * 1e4), 6000)
ein = np.log10(eddington_df_from_rgrid_rho(EPS, r, p["rho2"] * einasto_shape(r, p["r2"], p["alpha"]))[0])

## Export

In [ ]:
medians, residuals = [EPS], [EPS]
for name, c in curves.items():
    np.savetxt(out / f"fig3_{name}.dat", np.column_stack([EPS, c.T]), fmt="%.8g")
    med = np.median(c, axis=0)
    medians.append(med)
    residuals += [med - ein, np.std(c, axis=0)]
np.savetxt(out / "fig3_medians.dat", np.column_stack(medians), fmt="%.8g", header="eps raw virial rs")
np.savetxt(out / "fig3_einasto.dat", np.column_stack([EPS, ein]), fmt="%.8g", header="eps log10f_einasto")
np.savetxt(out / "fig4.dat", np.column_stack(residuals), fmt="%.8g",
           header="eps res_raw std_raw res_virial std_virial res_rs std_rs")